# 🛠️ Notebook 2: Restaurant — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/restaurant
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from itertools import count

@dataclass(frozen=True)
class MenuItem:
    name: str
    price: float
    category: str

class Menu:
    def __init__(self, items: list[MenuItem]):
        self.items = {i.name: i for i in items}
    def find(self, name: str) -> MenuItem:
        return self.items[name]
    def by_category(self, cat: str) -> list[MenuItem]:
        return [i for i in self.items.values() if i.category == cat]

class TableState(Enum):
    FREE = "free"; TAKEN = "taken"; RESERVED = "reserved"

@dataclass
class Table:
    number: int
    seats: int
    state: TableState = TableState.FREE


In [ ]:
class OrderStatus(Enum):
    OPEN = "open"; PLACED = "placed"; SERVED = "served"; PAID = "paid"

@dataclass
class OrderItem:
    item: MenuItem
    qty: int = 1
    notes: str = ""
    def line_total(self) -> float:
        return self.item.price * self.qty

_oids = count(1)

@dataclass
class Order:
    table: Table
    id: int = field(default_factory=lambda: next(_oids))
    items: list[OrderItem] = field(default_factory=list)
    status: OrderStatus = OrderStatus.OPEN

    def add(self, item: MenuItem, qty=1, notes=""):
        self._require(OrderStatus.OPEN)
        self.items.append(OrderItem(item, qty, notes))

    def place(self):  self._require(OrderStatus.OPEN);   self.status = OrderStatus.PLACED
    def serve(self):  self._require(OrderStatus.PLACED); self.status = OrderStatus.SERVED
    def subtotal(self) -> float: return sum(i.line_total() for i in self.items)

    def bill(self, tax_rate=0.08, tip_rate=0.15) -> dict:
        sub = self.subtotal()
        tax, tip = round(sub * tax_rate, 2), round(sub * tip_rate, 2)
        total = round(sub + tax + tip, 2)
        return {"subtotal": sub, "tax": tax, "tip": tip, "total": total}

    def pay(self):
        self._require(OrderStatus.SERVED)
        self.status = OrderStatus.PAID
        return self.bill()

    def _require(self, s):
        if self.status != s: raise ValueError(f"need {s}, got {self.status}")


In [ ]:
menu = Menu([
    MenuItem("Margherita",  10, "pizza"),
    MenuItem("Pepperoni",   12, "pizza"),
    MenuItem("Coke",         3, "drinks"),
    MenuItem("Tiramisu",     6, "dessert"),
])

t5 = Table(number=5, seats=4, state=TableState.TAKEN)
order = Order(table=t5)
order.add(menu.find("Margherita"))
order.add(menu.find("Coke"), qty=2)
order.add(menu.find("Tiramisu"), notes="no dusting")
order.place()
order.serve()
print("Bill for table", t5.number, "→", order.pay())

# Invalid transition
try: order.place()
except ValueError as e: print("expected:", e)


### Try it
- Add `Waiter`, `Chef`, `Manager` as `Staff` subclasses; attach waiter to Order.
- Add a `Kitchen` that receives placed orders and returns cook time estimates.
- Introduce `Reservation` (future booking) that transitions a Table from RESERVED → TAKEN.